# Step 1 — Recovering a pattern shift from two frames

**SMOTS method · deflectometry bench · SOLAR Lab, Arizona State University**

This notebook works through the first step of the SMOTS method in one dimension.
It is written to be run cell by cell in Google Colab.

**What the step does.** A fringe pattern (a sinusoid) is displayed on a screen. The camera
records it twice: once as a reference, and once after a mirror has tilted slightly. A tilt moves
the reflected pattern sideways. The quantity we need is **how far the pattern moved**, because
that distance gives the mirror tilt.

**What is computed here**

| Symbol | Meaning | Units |
|---|---|---|
| f(x) | reference pattern | grey level |
| f(x + Δ) | pattern after the mirror tilted | grey level |
| Δ | shift of the pattern | pixels, then mm |
| ψ | phase of the shift | radians |
| θ | mirror tilt | microradians |

Reference: H. Choi, I. Trumper, M. Dubin, W. Zhao, D. W. Kim, *Proc. SPIE* **10377**, 103770G (2017).

## 1. Libraries

Two libraries are used.

- **NumPy** (`numpy`) handles arrays of numbers and the Fourier transform. `np.cos`, `np.fft.fft`
  and `np.arctan2` all come from it. It is the only library the method itself needs.
- **Matplotlib** (`matplotlib.pyplot`) draws the plots. It is used here only to see what the
  patterns and the spectrum look like; it takes no part in the calculation.

Both are pre-installed in Google Colab, so no installation step is required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("numpy", np.__version__)

## 2. Parameters of the bench

These five numbers describe the screen and the geometry. They are constants: the method does not
change them, it only uses them to convert the result into physical units.

| Name in the code | Meaning | Value |
|---|---|---|
| `SCREEN_WIDTH_MM` | physical width of the screen | 344 mm |
| `SCREEN_PIXELS` | horizontal pixel count of the screen | 3840 |
| `PIXEL_PITCH_MM` | width of one screen pixel = 344 / 3840 | 0.0896 mm |
| `PERIOD_PX` | fringe period: screen pixels per fringe | 30 px |
| `Z_D_MM` | distance from the mirror to the screen | 400 mm |

`N` is the number of samples along the line we analyse. It is set to 17 × 30 = 510, a **whole
number of fringe periods**. The reason appears in section 5.

In [ ]:
SCREEN_WIDTH_MM = 344.0
SCREEN_PIXELS   = 3840
PIXEL_PITCH_MM  = SCREEN_WIDTH_MM / SCREEN_PIXELS   # 0.0896 mm per screen pixel
PERIOD_PX       = 30.0                              # one fringe every 30 screen pixels
Z_D_MM          = 400.0                             # mirror-to-screen distance

PERIODS_IN_WINDOW = 17
N = int(PERIODS_IN_WINDOW * PERIOD_PX)              # 510 samples

print(f"pixel pitch   = {PIXEL_PITCH_MM*1000:.1f} um")
print(f"fringe period = {PERIOD_PX:.0f} px = {PERIOD_PX*PIXEL_PITCH_MM:.3f} mm")
print(f"samples N     = {N}")

## 3. Building the two patterns

The patterns are generated, not photographed. That is deliberate: if we choose the shift
ourselves, we know the correct answer in advance and can check whether the method returns it.

The pattern is

$$ f(x) = 128 + 100 \cos\left(\frac{2\pi (x + \Delta)}{P}\right) $$

- **128** is the mean brightness. A camera image holds values from 0 to 255, so the sinusoid is
  centred in the middle of that range. This constant level is called the **DC level** — a term
  borrowed from electronics, where DC means a signal that does not vary. Here it is the constant
  background brightness on which the fringes sit.
- **100** is the amplitude, so the values run between 28 and 228 and never clip at 0 or 255.
- **P = 30** makes one complete fringe every 30 samples.
- **Δ (`shift_px`)** moves the whole pattern sideways. This is what a mirror tilt does to the
  reflected pattern, so shifting the sinusoid stands in for tilting a mirror.

In [ ]:
def displayed_pattern(x, shift_px=0.0):
    """Fringe pattern sampled along a line, optionally shifted sideways."""
    return 128.0 + 100.0 * np.cos(2 * np.pi * (x + shift_px) / PERIOD_PX)


x         = np.arange(N, dtype=float)      # sample positions 0, 1, 2, ... 509
reference = displayed_pattern(x)           # the "before" frame
measured  = displayed_pattern(x, 3.0)      # the "after" frame, shifted by 3 px

plt.figure(figsize=(10, 3))
plt.plot(x[:120], reference[:120], label="reference  f(x)")
plt.plot(x[:120], measured[:120],  label="measured  f(x + 3 px)")
plt.xlabel("sample"); plt.ylabel("grey level"); plt.legend(); plt.title("First 120 samples")
plt.show()

## 4. Removing the DC level and subtracting

Two operations, in this order.

1. **Remove the DC level.** Subtract the mean of each pattern. What remains is the fringe
   variation alone, centred on zero. The constant background carries no information about the
   shift, and leaving it in would place a large value at frequency zero in the next step.
2. **Subtract the two patterns.** The difference f(x + Δ) − f(x) is called the **sheared
   pattern**. Note what it is: still a sinusoid of the same frequency, but its amplitude depends
   on how far the pattern moved. A small shift gives a small difference; a large shift gives a
   large one. That is the whole basis of the method.

In [ ]:
ref     = reference - reference.mean()               # DC level removed
sheared = (measured - measured.mean()) - ref         # the sheared pattern

plt.figure(figsize=(10, 3))
plt.plot(x[:120], ref[:120],     label="reference, DC removed")
plt.plot(x[:120], sheared[:120], label="sheared = measured - reference")
plt.xlabel("sample"); plt.ylabel("grey level"); plt.legend()
plt.title("The difference is a sinusoid whose amplitude grows with the shift")
plt.show()

## 5. Fourier transform of both patterns

The Fourier transform rewrites a signal as a sum of sinusoids, and reports how much of each
frequency is present and at what phase. `np.fft.fft` returns one complex number per frequency:
its **magnitude** is how strong that frequency is, its **angle** is that frequency's phase.

Our pattern contains essentially one frequency — the fringe — so the transform shows one strong
peak. The index of that peak is called the **carrier index**, written `k`.

**Why N must hold a whole number of periods.** The transform assumes the signal repeats endlessly.
With 17 complete fringes in the window, the end joins the beginning smoothly. If the window ended
mid-fringe, the join would be a step, and the energy of the fringe would spread across
neighbouring indices instead of landing on one. That spreading is called **spectral leakage**.

In [ ]:
F = np.fft.fft(ref)        # transform of the reference
S = np.fft.fft(sheared)    # transform of the sheared pattern

k = np.argmax(np.abs(F[1:N // 2])) + 1     # index of the strongest frequency
print("carrier index k =", k, " (expected 17: seventeen fringes in the window)")

plt.figure(figsize=(10, 3))
plt.plot(np.abs(F[:60]), marker="o", ms=3)
plt.axvline(k, color="red", ls="--", label=f"carrier, k = {k}")
plt.xlabel("frequency index"); plt.ylabel("magnitude"); plt.legend()
plt.title("Spectrum of the reference pattern")
plt.show()

## 6. Extracting the phase

At the carrier index, the two transforms are related by

$$ \frac{S}{F} = e^{i\psi} - 1, \qquad \psi = \frac{2\pi \Delta}{P} $$

Dividing removes everything about the pattern itself — its brightness, its contrast, the camera
scale — and leaves only the term containing the shift.

The ratio is a complex number. Writing it in parts: its real part is cos ψ − 1 and its imaginary
part is sin ψ. Adding 1 to the real part recovers cos ψ, and then

$$ \psi = \mathrm{atan2}(\mathrm{Im}, \; 1 + \mathrm{Re}) $$

returns the phase **with its sign**, so a shift to the left and a shift to the right can be told
apart.

The published form of the equation takes the magnitude instead:
ψ = arccos(1 − |S/F|² / 2). That returns the same size but is always positive, so the direction
of the shift is lost. Both are computed below for comparison.

In [ ]:
ratio = S[k] / F[k]                                   # = e^(i psi) - 1

psi_signed = np.arctan2(ratio.imag, 1 + ratio.real)   # phase with its sign
psi_paper  = np.arccos(np.clip(1 - abs(ratio)**2 / 2, -1, 1))   # published form, no sign

shift_px = psi_signed / (2 * np.pi) * PERIOD_PX       # phase -> pixels

print(f"ratio      = {ratio:.6f}")
print(f"psi signed = {psi_signed:+.6f} rad")
print(f"psi paper  = {psi_paper:+.6f} rad   (same size, always positive)")
print(f"recovered shift = {shift_px:+.6f} px     (3.000 was applied)")

## 7. The complete function

The steps above, collected into one function. This is the form used for the tests that follow.

In [ ]:
def recover_shift_px(reference, measured):
    """Return the shift between two patterns, in screen pixels (signed)."""
    ref     = reference - reference.mean()            # remove the DC level
    sheared = (measured - measured.mean()) - ref      # the sheared pattern

    F = np.fft.fft(ref)
    S = np.fft.fft(sheared)

    k     = np.argmax(np.abs(F[1:N // 2])) + 1        # carrier index
    ratio = S[k] / F[k]                               # = e^(i psi) - 1

    psi_signed = np.arctan2(ratio.imag, 1 + ratio.real)
    psi_paper  = np.arccos(np.clip(1 - abs(ratio)**2 / 2, -1, 1))

    return (psi_signed / (2*np.pi)) * PERIOD_PX, (psi_paper / (2*np.pi)) * PERIOD_PX

## 8. Test 1 — recovering shifts that are already known

Five shifts are applied, including negative ones, and compared with what the function returns.
The last column is the published form, kept to show the loss of direction.

In [ ]:
print(" applied (px)   recovered (px)      error (px)   published form (px)")
for true_px in (0.5, 2.0, -3.5, 7.0, -11.0):
    got, paper = recover_shift_px(reference, displayed_pattern(x, true_px))
    print(f"   {true_px:+8.2f}      {got:+10.4f}     {got-true_px:+10.2e}        {paper:+8.2f}")

**Reading the table.** The error column is of order 10⁻¹⁶, which is the precision of the computer's
own arithmetic — the method is exact for data without noise. In the last column, −3.5 and −11.0
appear as +3.5 and +11.0: the published form gives the size of the shift but not its direction.

## 9. Test 2 — finding where the method stops working

The applied shift is increased past half a fringe period (15 px) to see what happens.

In [ ]:
applied   = np.arange(-32, 32.5, 1.0)
recovered = np.array([recover_shift_px(reference, displayed_pattern(x, t))[0] for t in applied])

plt.figure(figsize=(7, 5))
plt.plot(applied, applied, "--", color="grey", label="ideal (recovered = applied)")
plt.plot(applied, recovered, "o", ms=4, label="recovered")
plt.axvline( PERIOD_PX/2, color="red", ls=":", label="half a period, +/- 15 px")
plt.axvline(-PERIOD_PX/2, color="red", ls=":")
plt.xlabel("applied shift (px)"); plt.ylabel("recovered shift (px)"); plt.legend()
plt.title("Recovery is correct only within half a fringe period")
plt.show()

for t in (14.0, 15.0, 16.0, 20.0, 31.0):
    got, _ = recover_shift_px(reference, displayed_pattern(x, t))
    print(f" applied {t:+6.1f} px   recovered {got:+8.3f} px")

**Reading the plot.** Up to ±15 px the points sit on the dashed line. Past that they fold back:
+16 px is reported as −14 px, which is 16 − 30. A shift cannot be distinguished from the same
shift minus one whole fringe period, because both produce the same pattern. The usable range is
therefore ±P/2.

## 10. Converting the shift into a mirror tilt

A mirror tilted by θ turns the reflected ray by 2θ, so a tilt moves the pattern on the screen by

$$ \Delta = 2\,\theta\,z_d \qquad \Longrightarrow \qquad \theta = \frac{\Delta}{2 z_d} $$

with z_d the mirror-to-screen distance. The shift is first converted from pixels to millimetres
using the screen pixel pitch.

In [ ]:
print("  shift (px)     shift (mm)     mirror tilt (urad)")
for shift_px in (0.1, 1.0, PERIOD_PX/2):
    shift_mm   = shift_px * PIXEL_PITCH_MM
    theta_urad = shift_mm / (2 * Z_D_MM) * 1e6
    print(f"   {shift_px:7.2f}       {shift_mm:8.4f}         {theta_urad:9.2f}")

print(f"\nUsable range: +/- {(PERIOD_PX/2)*PIXEL_PITCH_MM/(2*Z_D_MM)*1e6:.0f} urad "
      f"with a {PERIOD_PX:.0f} px fringe at z_d = {Z_D_MM:.0f} mm")

## 11. Summary

| Result | Value |
|---|---|
| Recovery of known shifts | exact to the precision of the arithmetic |
| Usable range | ±15 px, that is ±P/2 |
| Angular resolution | 0.1 px of shift = 11.2 µrad of mirror tilt |
| Angular range | ±1,680 µrad |

**Open items**

1. The script uses z_d = 400 mm; the bench setting in the Week 3 study is 300 mm. The two must be
   reconciled before any angle is quoted.
2. The paper prints the shift theorem with a negative exponent, where the standard transform used
   by NumPy gives a positive one. Magnitudes are unaffected, but the sign of the reported tilt
   changes.
3. The fringe period sets the range: 30 px gives ±1,680 µrad. The range the experiment requires
   has not yet been defined.

**Next step.** The same recovery applied to a two-dimensional image, with a mask around a single
mirror. That introduces the digital apertures and spectral leakage, neither of which appears in
the one-dimensional case.

## 12. Exporting the code

The cell below writes the working parts of this notebook to a plain Python file,
`step1_shear_recovery.py`, which can be downloaded from the Colab file browser and run with
`python step1_shear_recovery.py`.

In [ ]:
source = '''"""Step 1 - recovery of a pattern shift from two frames (SMOTS method)."""

import numpy as np

SCREEN_WIDTH_MM = 344.0
SCREEN_PIXELS   = 3840
PIXEL_PITCH_MM  = SCREEN_WIDTH_MM / SCREEN_PIXELS
PERIOD_PX       = 30.0
Z_D_MM          = 400.0
N               = int(17 * PERIOD_PX)


def displayed_pattern(x, shift_px=0.0):
    return 128.0 + 100.0 * np.cos(2 * np.pi * (x + shift_px) / PERIOD_PX)


def recover_shift_px(reference, measured):
    ref     = reference - reference.mean()
    sheared = (measured - measured.mean()) - ref
    F = np.fft.fft(ref)
    S = np.fft.fft(sheared)
    k     = np.argmax(np.abs(F[1:N // 2])) + 1
    ratio = S[k] / F[k]
    psi_signed = np.arctan2(ratio.imag, 1 + ratio.real)
    psi_paper  = np.arccos(np.clip(1 - abs(ratio) ** 2 / 2, -1, 1))
    return (psi_signed / (2 * np.pi)) * PERIOD_PX, (psi_paper / (2 * np.pi)) * PERIOD_PX


if __name__ == "__main__":
    x = np.arange(N, dtype=float)
    reference = displayed_pattern(x)

    print("TEST 1 - known shifts")
    for true_px in (0.5, 2.0, -3.5, 7.0, -11.0):
        got, paper = recover_shift_px(reference, displayed_pattern(x, true_px))
        print(f"  applied {true_px:+7.2f}  recovered {got:+10.4f}  published {paper:+7.2f}")

    print("\\\\nTEST 2 - beyond half a period")
    for true_px in (14.0, 15.0, 16.0, 20.0, 31.0):
        got, _ = recover_shift_px(reference, displayed_pattern(x, true_px))
        print(f"  applied {true_px:+7.2f}  recovered {got:+10.3f}")

    print("\\\\nSHIFT TO MIRROR TILT")
    for shift_px in (0.1, 1.0, PERIOD_PX / 2):
        theta_urad = shift_px * PIXEL_PITCH_MM / (2 * Z_D_MM) * 1e6
        print(f"  {shift_px:6.2f} px -> {theta_urad:8.2f} urad")
'''

with open("step1_shear_recovery.py", "w") as fh:
    fh.write(source)

print("written: step1_shear_recovery.py")